# LLM12: LLM Inference — Decoding Strategies & KV-Cache

## Lab Overview

This lab explores how Large Language Models generate text at inference time. We cover the two-stage inference pipeline (**Prefill** and **Decode**), the **arithmetic intensity** that makes decode memory-bound, decoding strategies (greedy, sampling with temperature/top-k/top-p), the critical **KV-Cache** optimization, real model inference with **Qwen3-0.6B**, and **Multi-Token Prediction (MTP)** for faster generation.

> **Quick terms:** **Prefill** runs the model once on the entire prompt (many tokens in parallel). **Decode** generates **one new token at a time** autoregressively. **KV-Cache** stores past **key/value** tensors from attention so earlier tokens are not recomputed each step. **MTP** predicts multiple future tokens in a single forward pass.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

Understand LLM inference from fundamentals to advanced optimizations.

1. **Describe the Two-Stage Pipeline**: Prefill (process entire prompt) and Decode (generate token by token), and why decode is memory-bound.
2. **Analyze Arithmetic Intensity**: Compute FLOPs vs bytes loaded for prefill and decode to understand hardware bottlenecks.
3. **Implement Decoding Strategies**: Greedy, Top-K sampling, Top-P (nucleus) sampling, and temperature scaling.
4. **Understand Repetition Penalties**: How `repetition_penalty`, `presence_penalty`, and `frequency_penalty` work.
5. **Implement KV-Cache**: Cache Key/Value tensors across decoding steps to avoid recomputation.
6. **Analyze KV-Cache Memory**: How GQA reduces KV-Cache, and why long contexts make KV-Cache the bottleneck.
7. **Real Model Inference**: Run prefill/decode on Qwen3-0.6B and observe KV-Cache in a real GQA model.
8. **Multi-Token Prediction (MTP)**: Predict multiple future tokens per forward pass to accelerate decode.

---


## 1. Environment Setup


In [6]:
import math
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.10.0+rocm7.1
GPU: Radeon RX 7900 XTX
GPU Memory: 25.8 GB


## 2. LLM Inference Pipeline Overview

An LLM takes a token sequence as input, passes it through $N$ Decoder layers, and outputs **logits** (unnormalized scores over the vocabulary). A **decoding strategy** converts logits into the next token.

### Two-Stage Inference:

| Stage       | Input                   | Output                             | Characteristic                                  |
| ----------- | ----------------------- | ---------------------------------- | ----------------------------------------------- |
| **Prefill** | Full prompt (s tokens)  | Logits for next token + cached K,V | Compute-bound: processes all tokens in parallel |
| **Decode**  | One new token at a time | Next token logits                  | Memory-bound: sequential, uses KV-Cache         |

### Memory Access Analysis (Naive, per Decoder Layer)

Understanding why Decode is memory-bound requires analyzing how much data is **loaded from memory** per step.

#### Model Weight Access:

Each Decoder layer has attention weights ($W_Q, W_K, W_V, W_O$) and MLP weights ($W_{up}, W_{gate}, W_{down}$).
For hidden dim $h$, the total weight per layer is approximately $\sim 16h^2$ parameters (with SwiGLU MLP).

| Stage | Tokens processed | Weight loads | Weights per token |
| ----- | ---------------- | ------------ | ----------------- |
| **Prefill** | $s$ tokens (parallel) | $16h^2 \times L$ bytes (once) | $\frac{16h^2 \times L}{s}$ |
| **Decode** | 1 token | $16h^2 \times L$ bytes (each step) | $16h^2 \times L$ |

**Key insight**: Decode loads the **entire model weights** for just **one token**.
For Qwen3-0.6B ($h$=1024, $L$=28): weights ≈ 1.2 GB. At 30 tok/s decode, that's 36 GB/s bandwidth just for weights.

#### KV-Cache Access:

| Stage | KV-Cache read per step |
| ----- | ---------------------- |
| **Prefill** | Write only (build cache) |
| **Decode** | Read all $s$ cached positions × $L$ layers × $H_{kv}$ heads × $d_{head}$ × 2(K+V) |

For Qwen3-0.6B at $s$=8192: KV read ≈ $28 \times 8 \times 128 \times 8192 \times 2 \times 2$ bytes ≈ 0.9 GB per decode step.

#### Arithmetic Intensity (FLOPs / Bytes loaded):

| Stage | FLOPs (per layer) | Bytes loaded | Arithmetic Intensity |
| ----- | ----------------- | ------------ | -------------------- |
| **Prefill** | $24bsh^2 + 4bs^2h$ | $16h^2 \times 2$ bytes (weights) + KV write | High ($\gg 1$) → **compute-bound** |
| **Decode** ($b$=1) | $24h^2 + 4sh$ | $16h^2 \times 2$ + $2 \times L \times H_{kv} \times s \times d_{head} \times 2$ | Low ($\ll 1$) → **memory-bound** |

> **Arithmetic intensity** = FLOPs / bytes loaded. When it's low, the GPU spends more time waiting for data than computing.
> Prefill processes many tokens in parallel → high intensity → GPU is busy.
> Decode processes 1 token → low intensity → GPU is idle, waiting for memory.


### FLOPs Analysis (per Decoder Layer):

- **Self-Attention**: $8bsh^2 + 4bs^2h$ (Q/K/V projections + attention + output projection)
- **MLP (4× expansion)**: $16bsh^2$
- **Total per layer**: $24bsh^2 + 4bs^2h$

Where $b$ = batch size, $s$ = sequence length, $h$ = hidden dimension.

> **Note:** These FLOPs formulas are simplified back-of-the-envelope estimates for teaching.
> Exact counts depend on implementation details such as bias terms, fused kernels, grouped-query attention (GQA), and the exact MLP expansion ratio.


In [7]:
# Arithmetic Intensity Analysis: Prefill vs Decode
# Arithmetic intensity = FLOPs / bytes loaded from memory
# Low intensity → memory-bound (GPU waits for data)
# High intensity → compute-bound (GPU is busy computing)

def arithmetic_intensity_prefill(s, h, num_layers, kv_heads, head_dim, dtype_bytes=2):
    """Compute arithmetic intensity for Prefill stage (one forward pass on s tokens)."""
    # FLOPs per layer: 24*b*s*h^2 + 4*b*s^2*h (b=1)
    flops_per_layer = 24 * s * h**2 + 4 * s**2 * h
    total_flops = flops_per_layer * num_layers

    # Bytes loaded: model weights (once) + KV write
    # Weights: ~16*h^2 params per layer (Q/K/V/O + MLP up/gate/down)
    weight_bytes = 16 * h**2 * num_layers * dtype_bytes
    # KV write: 2 * num_layers * kv_heads * head_dim * s * 2(K+V) * dtype
    kv_write_bytes = 2 * num_layers * kv_heads * head_dim * s * 2 * dtype_bytes
    total_bytes = weight_bytes + kv_write_bytes

    intensity = total_flops / total_bytes
    return total_flops, total_bytes, intensity


def arithmetic_intensity_decode(s, h, num_layers, kv_heads, head_dim, dtype_bytes=2):
    """Compute arithmetic intensity for Decode stage (one token, with KV-Cache of length s)."""
    # FLOPs per layer: 24*h^2 + 4*s*h (b=1, one new token)
    flops_per_layer = 24 * h**2 + 4 * s * h
    total_flops = flops_per_layer * num_layers

    # Bytes loaded: model weights (every step!) + KV read (all cached)
    weight_bytes = 16 * h**2 * num_layers * dtype_bytes
    kv_read_bytes = 2 * num_layers * kv_heads * head_dim * s * 2 * dtype_bytes
    total_bytes = weight_bytes + kv_read_bytes

    intensity = total_flops / total_bytes
    return total_flops, total_bytes, intensity


# Qwen3-0.6B config
h = 1024
num_layers = 28
kv_heads = 8
head_dim = 128

print("=== Arithmetic Intensity: Prefill vs Decode (Qwen3-0.6B) ===")
print("Arithmetic intensity = FLOPs / bytes loaded. Higher = more compute-bound.\n")
print(f"{'Stage':<10s} | {'Seq Len':>8s} | {'FLOPs':>14s} | {'Bytes Loaded':>14s} | {'Intensity':>10s} | {'Bottleneck':>12s}")
print("-" * 80)

for s in [128, 512, 2048, 8192, 32768]:
    # Prefill
    pf_flops, pf_bytes, pf_intensity = arithmetic_intensity_prefill(s, h, num_layers, kv_heads, head_dim)
    pf_bound = "compute" if pf_intensity > 1 else "memory"
    print(f"{'Prefill':<10s} | {s:>8,d} | {pf_flops:>14,.0f} | {pf_bytes:>12,.0f} B | {pf_intensity:>9.2f} | {pf_bound:>12s}")

    # Decode
    dc_flops, dc_bytes, dc_intensity = arithmetic_intensity_decode(s, h, num_layers, kv_heads, head_dim)
    dc_bound = "compute" if dc_intensity > 1 else "memory"
    print(f"{'Decode':<10s} | {s:>8,d} | {dc_flops:>14,.0f} | {dc_bytes:>12,.0f} B | {dc_intensity:>9.2f} | {dc_bound:>12s}")
    print()

print("Key observations:")
print("  1. Prefill intensity >> 1 -> compute-bound (GPU is busy)")
print("  2. Decode intensity << 1 -> memory-bound (GPU waits for data)")
print("  3. Decode loads entire model weights for just 1 token -> very inefficient")
print("  4. As seq_len grows, Decode gets even worse (more KV-Cache to read)")

=== Arithmetic Intensity: Prefill vs Decode (Qwen3-0.6B) ===
Arithmetic intensity = FLOPs / bytes loaded. Higher = more compute-bound.

Stage      |  Seq Len |          FLOPs |   Bytes Loaded |  Intensity |   Bottleneck
--------------------------------------------------------------------------------
Prefill    |      128 | 92,073,361,408 |  968,884,224 B |     95.03 |      compute
Decode     |      128 |    719,323,136 |  968,884,224 B |      0.74 |       memory

Prefill    |      512 | 390,842,023,936 | 1,056,964,608 B |    369.78 |      compute
Decode     |      512 |    763,363,328 | 1,056,964,608 B |      0.72 |       memory

Prefill    |    2,048 | 1,924,145,348,608 | 1,409,286,144 B |   1365.33 |      compute
Decode     |    2,048 |    939,524,096 | 1,409,286,144 B |      0.67 |       memory

Prefill    |    8,192 | 13,469,017,440,256 | 2,818,572,288 B |   4778.67 |      compute
Decode     |    8,192 |  1,644,167,168 | 2,818,572,288 B |      0.58 |       memory

Prefill    |   32

## 3. Decoding Strategies

Given logits $z \in \mathbb{R}^{|V|}$ from the LM head, we need to pick the next token.

### Temperature Scaling

$$\tilde{p}_i = \text{softmax}(z_i / T)$$

- $T < 1$: sharper distribution (more deterministic)
- $T > 1$: flatter distribution (more random)
- As $T \to 0^+$, sampling approaches greedy decoding
- As $T \to \infty$, the softmax distribution approaches uniform over the candidate tokens

### Top-K Sampling

Keep only the $k$ highest-probability tokens, zero out the rest, re-normalize.

### Top-P (Nucleus) Sampling

Keep the smallest set of tokens whose cumulative probability $\geq p$, zero out the rest.

**Typical pipeline**: Temperature → Top-K → Top-P → Sample


In [8]:
def greedy_decode(logits: torch.Tensor) -> int:
    """Select the token with the highest logit."""
    return logits.argmax(dim=-1).item()


def temperature_scale(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """Scale logits by temperature."""
    if temperature <= 0:
        raise ValueError("Temperature must be > 0")
    return logits / temperature


def top_k_filter(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Set all non-top-k logits to -inf so they receive zero probability after softmax."""
    if k <= 0 or k >= logits.size(-1):
        return logits
    topk_vals, _ = torch.topk(logits, k)
    threshold = topk_vals[..., -1].unsqueeze(-1)
    return logits.masked_fill(logits < threshold, float("-inf"))


def top_p_filter(logits: torch.Tensor, p: float) -> torch.Tensor:
    """Keep the smallest set of tokens whose cumulative probability reaches p."""
    if not (0.0 < p <= 1.0):
        raise ValueError("top_p must be in (0, 1].")

    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cumulative_probs = sorted_probs.cumsum(dim=-1)

    # Remove tokens whose inclusion would push cumulative probability past p,
    # while always keeping at least the first token.
    mask = cumulative_probs - sorted_probs >= p
    sorted_logits = sorted_logits.masked_fill(mask, float("-inf"))

    output = logits.clone()
    output.scatter_(-1, sorted_idx, sorted_logits)
    return output


def sample_token(logits: torch.Tensor, temperature=1.0, top_k=0, top_p=1.0) -> int:
    """Full sampling pipeline: temperature -> top_k -> top_p -> sample."""
    logits = temperature_scale(logits, temperature)
    logits = top_k_filter(logits, top_k)
    logits = top_p_filter(logits, top_p)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# Demonstrate with synthetic logits
vocab_size = 10
logits = torch.tensor([2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5, -2.0, -3.0])
labels = [f"tok_{i}" for i in range(vocab_size)]

print("=== Decoding Strategy Comparison ===")
print(f"Logits: {logits.tolist()}")
print(f"Softmax probs: {torch.softmax(logits, -1).tolist()}")
print(f"\nGreedy: token {greedy_decode(logits)} (always highest)")

# Sample 20 times with different strategies
for label, kwargs in [
    ("T=0.3 (sharp)", {"temperature": 0.3}),
    ("T=1.0 (default)", {"temperature": 1.0}),
    ("T=2.0 (flat)", {"temperature": 2.0}),
    ("Top-K=3", {"top_k": 3}),
    ("Top-P=0.8", {"top_p": 0.8}),
    ("T=0.7+K=5+P=0.9", {"temperature": 0.7, "top_k": 5, "top_p": 0.9}),
]:
    samples = [sample_token(logits.clone(), **kwargs) for _ in range(20)]
    unique = sorted(set(samples))
    print(f"  {label:<20s}: unique={unique}, samples={samples}")

=== Decoding Strategy Comparison ===
Logits: [2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5, -2.0, -3.0]
Softmax probs: [0.3968255817890167, 0.24068687856197357, 0.1459839791059494, 0.08854375779628754, 0.05370450019836426, 0.03257342800498009, 0.019756780937314034, 0.011983094736933708, 0.007268114015460014, 0.0026737896259874105]

Greedy: token 0 (always highest)
  T=0.3 (sharp)       : unique=[0, 1, 2], samples=[0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  T=1.0 (default)     : unique=[0, 1, 2, 3, 4, 6, 7], samples=[2, 1, 3, 6, 0, 0, 0, 1, 1, 7, 4, 0, 2, 0, 4, 2, 4, 0, 0, 1]
  T=2.0 (flat)        : unique=[0, 1, 2, 3, 4, 5, 6, 7, 9], samples=[5, 4, 7, 6, 9, 3, 6, 3, 0, 1, 5, 7, 1, 2, 2, 0, 4, 3, 1, 0]
  Top-K=3             : unique=[0, 1, 2], samples=[1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 2, 0, 1, 2, 0, 0]
  Top-P=0.8           : unique=[0, 1, 2, 3], samples=[0, 1, 2, 3, 3, 0, 0, 3, 1, 0, 3, 2, 2, 2, 0, 2, 1, 0, 0, 0]
  T=0.7+K=5+P=0.9     : unique=[0, 1, 2], sample

## 4. Repetition Penalty

Even with temperature and top-k/p, models can produce repetitive text. Penalty mechanisms modify logits based on previously generated tokens:

- **Repetition penalty** (HuggingFace-style): For each previously seen token type, adjust its logit once:
  divide by `penalty` if the logit is positive, or multiply by `penalty` if the logit is negative
- **Presence penalty** (OpenAI): Subtract a constant from any previously seen token's logit
- **Frequency penalty** (OpenAI): Subtract a value proportional to how many times the token appeared


In [9]:
def apply_repetition_penalty(logits: torch.Tensor, generated_ids: list, penalty: float = 1.2) -> torch.Tensor:
    """HuggingFace-style repetition penalty."""
    if penalty <= 0:
        raise ValueError("penalty must be > 0")

    logits = logits.clone()
    for token_id in set(generated_ids):
        if logits[token_id] > 0:
            logits[token_id] /= penalty
        else:
            logits[token_id] *= penalty
    return logits


def apply_frequency_presence_penalty(
    logits: torch.Tensor, generated_ids: list, frequency_penalty: float = 0.5, presence_penalty: float = 0.3
) -> torch.Tensor:
    """OpenAI-style frequency + presence penalty."""
    logits = logits.clone()
    from collections import Counter

    counts = Counter(generated_ids)
    for token_id, count in counts.items():
        logits[token_id] -= frequency_penalty * count
        logits[token_id] -= presence_penalty  # applied once per unique token
    return logits


# Demonstrate
logits = torch.tensor([3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5])
generated = [0, 0, 1, 0]  # token 0 appears 3 times

print("=== Repetition Penalty Demo ===")
print(f"Original logits:  {logits.tolist()}")
print(f"Generated so far: {generated}")

penalized_rep = apply_repetition_penalty(logits, generated, penalty=1.5)
print(f"After rep_penalty=1.5: {penalized_rep.tolist()}")

penalized_fp = apply_frequency_presence_penalty(logits, generated, frequency_penalty=0.5, presence_penalty=0.3)
print(f"After freq=0.5, pres=0.3: {penalized_fp.tolist()}")
print(f"\nToken 0 logit: {logits[0]:.1f} -> rep: {penalized_rep[0]:.2f}, fp: {penalized_fp[0]:.2f}")
print(
    f"Token 2 logit: {logits[2]:.1f} -> rep: {penalized_rep[2]:.2f} (unchanged), fp: {penalized_fp[2]:.2f} (unchanged)"
)

=== Repetition Penalty Demo ===
Original logits:  [3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5]
Generated so far: [0, 0, 1, 0]
After rep_penalty=1.5: [2.0, 1.6666666269302368, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5]
After freq=0.5, pres=0.3: [1.2000000476837158, 1.7000000476837158, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5]

Token 0 logit: 3.0 -> rep: 2.00, fp: 1.20
Token 2 logit: 2.0 -> rep: 2.00 (unchanged), fp: 2.00 (unchanged)


## 5. Prefill & Decode — Step by Step

Let's build a minimal Transformer decoder and trace the two-stage pipeline explicitly.

> **Teaching simplification:** The tiny model below omits positional embeddings / RoPE.
> That is fine for demonstrating prefill, decode, and KV-Cache mechanics, but real LLMs need positional information to represent token order.


In [10]:
class TinyDecoderLayer(nn.Module):
    """Simplified single decoder layer with proper KV-Cache.

    Uses separate Q/K/V projections so we can cache the PROJECTED K and V
    tensors. This is the correct pattern — during decode, only the new token
    goes through K/V projection; past tokens' K/V come from cache.
    """

    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        # Separate Q/K/V projections (not nn.MultiheadAttention)
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.SiLU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x, kv_cache=None):
        """
        Args:
            x: (B, T, D) — full prompt (prefill) or single new token (decode)
            kv_cache: (cached_K, cached_V) each (B, H, S_past, Dh), or None
        Returns:
            output (B, T, D), new_kv_cache
        """
        B, T, D = x.shape
        residual = x
        x_norm = self.norm1(x)

        # Project Q/K/V for CURRENT tokens only
        Q = self.q_proj(x_norm).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K_new = self.k_proj(x_norm).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V_new = self.v_proj(x_norm).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Append to cache (the key optimization: past K/V are NOT re-projected!)
        if kv_cache is not None:
            K = torch.cat([kv_cache[0], K_new], dim=2)
            V = torch.cat([kv_cache[1], V_new], dim=2)
        else:
            K, V = K_new, V_new

        new_cache = (K.detach(), V.detach())

        # Apply a causal mask during prefill, where multiple query positions are processed together.
        # During decode we feed exactly one new token (T=1), so there are no "future" positions
        # within the current query block, and the single query is allowed to attend to all cached past tokens.
        attn_out = F.scaled_dot_product_attention(Q, K, V, is_causal=(kv_cache is None and T > 1))

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D)
        x = residual + self.out_proj(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x, new_cache


class TinyLM(nn.Module):
    """Minimal language model with proper KV-Cache support."""

    def __init__(self, vocab_size, hidden_dim, num_heads, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.layers = nn.ModuleList([TinyDecoderLayer(hidden_dim, num_heads) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)

    def forward(self, input_ids, kv_caches=None):
        x = self.embedding(input_ids)
        new_caches = []
        for i, layer in enumerate(self.layers):
            cache = kv_caches[i] if kv_caches else None
            x, new_cache = layer(x, cache)
            new_caches.append(new_cache)
        x = self.norm(x)
        return self.lm_head(x), new_caches


# Use a LARGER model so KV-Cache savings are measurable over Python overhead
vocab_size = 1000
hidden_dim, num_heads, num_layers = 512, 8, 6
model = TinyLM(vocab_size, hidden_dim, num_heads, num_layers).to(device)
model.eval()
print(f"TinyLM: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Config: dim={hidden_dim}, heads={num_heads}, layers={num_layers}, vocab={vocab_size}")

TinyLM: 19,927,040 parameters
Config: dim=512, heads=8, layers=6, vocab=1000


In [11]:
# Two-stage generation demonstration


@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens=10, temperature=1.0, top_k=0):
    """
    Generate tokens using prefill + decode with KV-Cache.
    """
    input_ids = prompt_ids.unsqueeze(0).to(device)  # (1, seq_len)
    generated = prompt_ids.tolist()

    # === Stage 1: PREFILL ===
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    logits, kv_caches = model(input_ids, kv_caches=None)
    if device.type == "cuda":
        torch.cuda.synchronize()
    prefill_time = time.perf_counter() - t0

    # Get next token from last position
    next_logits = logits[0, -1, :]  # (vocab_size,)
    next_token = sample_token(next_logits, temperature=temperature, top_k=top_k)
    generated.append(next_token)

    print(f"Prefill: processed {input_ids.size(1)} tokens in {prefill_time * 1000:.2f} ms")

    # === Stage 2: DECODE ===
    decode_times = []
    for step in range(max_new_tokens - 1):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        new_input = torch.tensor([[next_token]], device=device)
        logits, kv_caches = model(new_input, kv_caches=kv_caches)

        if device.type == "cuda":
            torch.cuda.synchronize()
        decode_time = time.perf_counter() - t0
        decode_times.append(decode_time)

        next_logits = logits[0, -1, :]
        next_token = sample_token(next_logits, temperature=temperature, top_k=top_k)
        generated.append(next_token)

    avg_decode = np.mean(decode_times) * 1000 if decode_times else 0.0
    print(f"Decode:  generated {max_new_tokens} tokens, avg {avg_decode:.2f} ms/token")
    return generated


# Run generation
prompt = torch.tensor(list(range(1, 21)))  # 20-token prompt
print("=== Generation with KV-Cache ===")
output = generate(model, prompt, max_new_tokens=10, temperature=0.8, top_k=10)
print(f"Generated sequence length: {len(output)} (prompt={20} + generated={10})")

=== Generation with KV-Cache ===
Prefill: processed 20 tokens in 618.10 ms
Decode:  generated 10 tokens, avg 5.81 ms/token
Generated sequence length: 30 (prompt=20 + generated=10)


## 6. KV-Cache Memory Analysis

For each decoder layer, the KV-Cache stores:

$$\text{KV memory} = b \times L \times H \times s \times d_{\text{head}} \times 2 \times \text{dtype\_size}$$

where:

- $b$ = batch size
- $L$ = number of decoder layers
- $H$ = number of attention heads
- $s$ = cached sequence length
- $d_{\text{head}}$ = head dimension
- $\times 2$ accounts for both K and V


In [12]:
def kv_cache_memory(batch, layers, heads, seq_len, head_dim, dtype_bytes=2):
    """Compute KV-Cache memory in bytes. dtype_bytes=2 corresponds to FP16/BF16."""
    return batch * layers * heads * seq_len * head_dim * 2 * dtype_bytes


# LLaMA-3.2-1B config
cfg = {"batch": 1, "layers": 16, "heads": 32, "head_dim": 64}

print("=== KV-Cache Memory (LLaMA-1B-like, FP16) ===")
print(f"{'Seq Length':>10s} | {'KV Memory':>12s} | {'Per Layer':>12s}")
print("-" * 40)
for s in [512, 1024, 2048, 4096, 8192, 16384, 32768, 131072]:
    total = kv_cache_memory(seq_len=s, **cfg)
    per_layer = total / cfg["layers"]
    print(f"{s:>10,d} | {total / 1e6:>10.1f} MB | {per_layer / 1e6:>10.2f} MB")

print(f"\nWith 128K context: {kv_cache_memory(seq_len=131072, **cfg) / 1e9:.2f} GB of KV-Cache alone.")
print("This motivates sparse attention and KV compression techniques.")

=== KV-Cache Memory (LLaMA-1B-like, FP16) ===
Seq Length |    KV Memory |    Per Layer
----------------------------------------
       512 |       67.1 MB |       4.19 MB
     1,024 |      134.2 MB |       8.39 MB
     2,048 |      268.4 MB |      16.78 MB
     4,096 |      536.9 MB |      33.55 MB
     8,192 |     1073.7 MB |      67.11 MB
    16,384 |     2147.5 MB |     134.22 MB
    32,768 |     4295.0 MB |     268.44 MB
   131,072 |    17179.9 MB |    1073.74 MB

With 128K context: 17.18 GB of KV-Cache alone.
This motivates sparse attention and KV compression techniques.


> **Note:** The `generate()` helper prints prefill/decode timing information for illustration, so the benchmark output below will include those intermediate messages.


In [13]:
# Compare generation speed: with KV-Cache vs without (recompute full sequence each step)


@torch.no_grad()
def generate_no_cache(model, prompt_ids, max_new_tokens=10, temperature=1.0, top_k=0):
    """Generate WITHOUT KV-Cache — recomputes full sequence each step."""
    generated = prompt_ids.tolist()
    for _ in range(max_new_tokens):
        input_ids = torch.tensor([generated], device=device)
        logits, _ = model(input_ids, kv_caches=None)  # no cache — full recompute!
        next_logits = logits[0, -1, :]
        next_token = sample_token(next_logits, temperature=temperature, top_k=top_k)
        generated.append(next_token)
    return generated


# Benchmark with enough tokens to see the difference
prompt = torch.tensor(list(range(1, 101)))  # 100-token prompt
n_gen = 100

# Warm up GPU (first CUDA call has overhead)
_ = model(torch.tensor([[1]], device=device))
if device.type == "cuda":
    torch.cuda.synchronize()

# With KV-Cache
torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
out1 = generate(model, prompt, max_new_tokens=n_gen, temperature=0.8, top_k=10)
if device.type == "cuda":
    torch.cuda.synchronize()
time_cache = time.perf_counter() - t0

# Without KV-Cache
torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
out2 = generate_no_cache(model, prompt, max_new_tokens=n_gen, temperature=0.8, top_k=10)
if device.type == "cuda":
    torch.cuda.synchronize()
time_nocache = time.perf_counter() - t0

print("\n=== KV-Cache Speed Comparison ===")
print(f"Config: {n_gen} new tokens, {len(prompt)}-token prompt, dim={hidden_dim}, layers={num_layers}")
print(f"With KV-Cache:    {time_cache * 1000:.1f} ms total")
print(f"Without KV-Cache: {time_nocache * 1000:.1f} ms total")
if time_cache > 0:
    print(f"Speedup: {time_nocache / time_cache:.2f}×")
print("\nWhy? Without cache, each decode step re-runs K/V projections on ALL past tokens.")
print("With cache, only 1 new token goes through K/V projection per step.")

Prefill: processed 100 tokens in 4.20 ms
Decode:  generated 100 tokens, avg 2.12 ms/token

=== KV-Cache Speed Comparison ===
Config: 100 new tokens, 100-token prompt, dim=512, layers=6
With KV-Cache:    251.8 ms total
Without KV-Cache: 253.3 ms total
Speedup: 1.01×

Why? Without cache, each decode step re-runs K/V projections on ALL past tokens.
With cache, only 1 new token goes through K/V projection per step.


## 7. Long-Context KV-Cache Bottleneck & Solutions

As context length grows, the KV-Cache becomes the **dominant memory consumer** during inference. Let's analyze this with Qwen3-0.6B's actual config and explore solutions.

### Why KV-Cache dominates

For a model with $L$ layers, $H_{\text{kv}}$ KV heads, head dimension $d_{\text{head}}$, and sequence length $s$:

$$\text{KV memory} = 2 \times b \times L \times H_{\text{kv}} \times s \times d_{\text{head}} \times \text{dtype\_bytes}$$

The factor of 2 accounts for both K and V tensors. With GQA, $H_{\text{kv}} < H_{\text{q}}$, which already helps — but at long contexts (32K, 128K), KV-Cache still grows to many GB.

### Solutions

| Technique | Idea | Memory Reduction | Quality Impact |
| --------- | ---- | ---------------- | -------------- |
| **GQA** (Grouped-Query Attention) | Fewer KV heads than Q heads | $\times \frac{H_{\text{kv}}}{H_{\text{q}}}$ | Minimal (trained in) |
| **MQA** (Multi-Query Attention) | Single KV head | $\times \frac{1}{H_{\text{q}}}$ | Small degradation |
| **Quantized KV-Cache** | Store KV in FP8/INT8 | $\sim 2\times$ | Minimal |
| **Sliding Window** (Mistral) | Attend only to last $w$ tokens | Capped at $w$ | Loses long-range info |
| **Sparse Attention** (MInference, Quest) | Select important KV pages | $\sim 2$-$5\times$ | Small degradation |
| **KV-Cache Compression** (StreamingLLM) | Keep sinks + window | Fixed size | Loses middle context |
| **PagedAttention** (vLLM) | Block-based memory management | No waste, not less total | None |
| **MLA** (Multi-head Latent Attention) | Compress KV into latent space | $\sim 10\times$ | Minimal (trained in) |


In [ ]:
def kv_cache_memory(layers, kv_heads, seq_len, head_dim, batch=1, dtype_bytes=2):
    """KV-Cache memory in bytes. dtype_bytes=2 for FP16/BF16."""
    return 2 * batch * layers * kv_heads * seq_len * head_dim * dtype_bytes


def mha_kv_memory(layers, q_heads, seq_len, head_dim, batch=1, dtype_bytes=2):
    """KV-Cache if the model used MHA (all Q heads have their own KV)."""
    return 2 * batch * layers * q_heads * seq_len * head_dim * dtype_bytes


# Qwen3-0.6B config
qwen_cfg = {
    "layers": 28,
    "q_heads": 16,
    "kv_heads": 8,
    "head_dim": 128,
}

print("=== KV-Cache Memory: Qwen3-0.6B (GQA, 8 KV heads) ===")
print(f"{'Seq Len':>10s} | {'GQA (8 KV)':>12s} | {'MHA (16 KV)':>12s} | {'GQA Savings':>12s} | {'Ratio':>6s}")
print("-" * 62)

for s in [512, 2048, 8192, 16384, 32768, 131072]:
    gqa_mem = kv_cache_memory(seq_len=s, layers=qwen_cfg["layers"], kv_heads=qwen_cfg["kv_heads"], head_dim=qwen_cfg["head_dim"])
    mha_mem = mha_kv_memory(seq_len=s, layers=qwen_cfg["layers"], q_heads=qwen_cfg["q_heads"], head_dim=qwen_cfg["head_dim"])
    savings = (1 - gqa_mem / mha_mem) * 100
    ratio = mha_mem / gqa_mem
    print(f"{s:>10,d} | {gqa_mem / 1e6:>10.1f} MB | {mha_mem / 1e6:>10.1f} MB | {savings:>10.0f}% | {ratio:>5.1f}x")

print("\nGQA halves KV-Cache vs MHA, but 128K context still needs ~4.6 GB for KV alone.")
print("For batch serving (e.g., 16 users), that's ~73 GB — exceeding most single-GPU memory.")

=== KV-Cache Memory: Qwen3-0.6B (GQA, 8 KV heads) ===
   Seq Len |   GQA (8 KV) |  MHA (16 KV) |  GQA Savings |  Ratio
--------------------------------------------------------------
       512 |       58.7 MB |      117.4 MB |         50% |   2.0x
     2,048 |      234.9 MB |      469.8 MB |         50% |   2.0x
     8,192 |      939.5 MB |     1879.0 MB |         50% |   2.0x
    16,384 |     1879.0 MB |     3758.1 MB |         50% |   2.0x
    32,768 |     3758.1 MB |     7516.2 MB |         50% |   2.0x
   131,072 |    15032.4 MB |    30064.8 MB |         50% |   2.0x

GQA halves KV-Cache vs MHA, but 128K context still needs ~4.6 GB for KV alone.
For batch serving (e.g., 16 users), that's ~73 GB — exceeding most single-GPU memory.


In [ ]:
# Demonstrate quantized KV-Cache memory savings

print("=== KV-Cache with Different Precision (Qwen3-0.6B, seq_len=32768) ===")
print(f"{'Precision':>10s} | {'Bytes/Elem':>10s} | {'KV Memory':>12s} | {'vs FP16':>10s}")
print("-" * 50)

s = 32768
fp16_mem = kv_cache_memory(seq_len=s, dtype_bytes=2, layers=qwen_cfg["layers"], kv_heads=qwen_cfg["kv_heads"], head_dim=qwen_cfg["head_dim"])

for name, dtype_bytes in [("FP32", 4), ("FP16/BF16", 2), ("FP8/INT8", 1), ("INT4", 0.5)]:
    mem = kv_cache_memory(seq_len=s, dtype_bytes=dtype_bytes, layers=qwen_cfg["layers"], kv_heads=qwen_cfg["kv_heads"], head_dim=qwen_cfg["head_dim"])
    ratio = fp16_mem / mem if mem > 0 else float("inf")
    print(f"{name:>10s} | {dtype_bytes:>10.1f} | {mem / 1e6:>10.1f} MB | {ratio:>8.1f}x")

print("\nFP8 quantization is widely used in production (vLLM, TensorRT-LLM).")
print("It halves KV-Cache memory with negligible quality loss.")

=== KV-Cache with Different Precision (Qwen3-0.6B, seq_len=32768) ===
 Precision | Bytes/Elem |    KV Memory |    vs FP16
--------------------------------------------------
      FP32 |        4.0 |     7516.2 MB |      0.5x
 FP16/BF16 |        2.0 |     3758.1 MB |      1.0x
  FP8/INT8 |        1.0 |     1879.0 MB |      2.0x
      INT4 |        0.5 |      939.5 MB |      4.0x

FP8 quantization is widely used in production (vLLM, TensorRT-LLM).
It halves KV-Cache memory with negligible quality loss.


In [ ]:
# Batch serving scenario: KV-Cache memory vs batch size

print("=== Batch Serving Memory Pressure (Qwen3-0.6B, seq_len=8192, FP16) ===")
print(f"{'Batch Size':>10s} | {'KV-Cache':>12s} | {'Model Weights':>14s} | {'KV % of Total':>14s}")
print("-" * 58)

model_weights_bytes = 596_049_920 * 2  # FP16
s = 8192

for batch in [1, 2, 4, 8, 16, 32]:
    kv = kv_cache_memory(seq_len=s, batch=batch, layers=qwen_cfg["layers"], kv_heads=qwen_cfg["kv_heads"], head_dim=qwen_cfg["head_dim"])
    total = kv + model_weights_bytes
    kv_pct = kv / total * 100
    print(f"{batch:>10d} | {kv / 1e9:>10.2f} GB | {model_weights_bytes / 1e9:>12.2f} GB | {kv_pct:>12.1f}%")

print("\nAt batch=8, KV-Cache already exceeds model weights!")
print("At batch=32, KV-Cache is 4x the model size — this is why PagedAttention and")
print("KV compression are critical for production LLM serving.")

=== Batch Serving Memory Pressure (Qwen3-0.6B, seq_len=8192, FP16) ===
Batch Size |     KV-Cache |  Model Weights |  KV % of Total
----------------------------------------------------------
         1 |       0.94 GB |         1.19 GB |         44.1%
         2 |       1.88 GB |         1.19 GB |         61.2%
         4 |       3.76 GB |         1.19 GB |         75.9%
         8 |       7.52 GB |         1.19 GB |         86.3%
        16 |      15.03 GB |         1.19 GB |         92.7%
        32 |      30.06 GB |         1.19 GB |         96.2%

At batch=8, KV-Cache already exceeds model weights!
At batch=32, KV-Cache is 4x the model size — this is why PagedAttention and
KV compression are critical for production LLM serving.


## 8. Real Model Inference — Qwen3-0.6B

So far we have used a toy model to understand the mechanics. Now let's run the same prefill/decode pipeline on a **real** model: **Qwen3-0.6B** (596M parameters).

Key architecture details:

| Parameter        | Value  |
| ---------------- | ------ |
| Hidden dim       | 1024   |
| Layers           | 28     |
| **Q heads**      | 16     |
| **KV heads**     | 8      |
| Head dim         | 128    |
| Vocab size       | 151,936|
| Max context      | 40,960 |

> **Grouped-Query Attention (GQA):** Qwen3 uses 16 Q heads but only 8 KV heads.
> Each pair of Q heads shares one set of K/V, which **halves the KV-Cache size** compared to standard multi-head attention (MHA). This is the dominant architecture in 2025-2026 models (LLaMA 3, Mistral, Qwen3, etc.).


In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
qwen_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, trust_remote_code=True)
qwen_model = qwen_model.to(device).eval()

print(f"Model: {model_id}")
print(f"Parameters: {sum(p.numel() for p in qwen_model.parameters()):,}")
print(f"Q heads: {qwen_model.config.num_attention_heads}")
print(f"KV heads: {qwen_model.config.num_key_value_heads}")
print(f"GQA ratio: {qwen_model.config.num_attention_heads // qwen_model.config.num_key_value_heads}:1")
print(f"Max context: {qwen_model.config.max_position_embeddings:,} tokens")

[aiter] import [module_aiter_enum] under /home/luyzh/miniconda3/envs/torch/lib/python3.10/site-packages/aiter/jit/module_aiter_enum.so
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+rocm7.1).


Model: Qwen/Qwen3-0.6B
Parameters: 596,049,920
Q heads: 16
KV heads: 8
GQA ratio: 2:1
Max context: 40,960 tokens


In [15]:
@torch.no_grad()
def real_prefill_decode(model, tokenizer, prompt, max_new_tokens=30):
    """Run prefill + decode on a real HuggingFace model, measuring each stage."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    prompt_len = input_ids.shape[1]

    # === Stage 1: PREFILL ===
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    outputs = model(input_ids, use_cache=True)
    if device.type == "cuda":
        torch.cuda.synchronize()
    prefill_time = time.perf_counter() - t0

    past_kv = outputs.past_key_values
    next_logits = outputs.logits[0, -1, :]
    next_token = torch.argmax(next_logits).item()
    generated_ids = [next_token]

    # Measure KV-Cache size
    kv_bytes = 0
    for k, v in past_kv:
        kv_bytes += k.numel() * k.element_size() + v.numel() * v.element_size()

    print(f"Prefill: {prompt_len} tokens in {prefill_time * 1000:.1f} ms")
    print(f"KV-Cache: {kv_bytes / 1e6:.2f} MB ({len(past_kv)} layers)")
    k0, v0 = past_kv[0]
    print(f"KV shape per layer: K={k0.shape}, V={v0.shape}")
    print(f"  (batch={k0.shape[0]}, kv_heads={k0.shape[1]}, seq_len={k0.shape[2]}, head_dim={k0.shape[3]})")

    # === Stage 2: DECODE ===
    decode_times = []
    for _ in range(max_new_tokens - 1):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        new_input = torch.tensor([[next_token]], device=device)
        outputs = model(new_input, past_key_values=past_kv, use_cache=True)

        if device.type == "cuda":
            torch.cuda.synchronize()
        decode_times.append(time.perf_counter() - t0)

        past_kv = outputs.past_key_values
        next_logits = outputs.logits[0, -1, :]
        next_token = torch.argmax(next_logits).item()
        generated_ids.append(next_token)

    avg_decode = np.mean(decode_times) * 1000
    total_decode = sum(decode_times) * 1000
    print(f"\nDecode: {max_new_tokens} tokens, avg {avg_decode:.1f} ms/token, total {total_decode:.1f} ms")
    print(f"Throughput: {max_new_tokens / (total_decode / 1000):.1f} tokens/s")

    output_text = tokenizer.decode(generated_ids)
    print(f"\nGenerated: {output_text}")
    return generated_ids


prompt = "The future of artificial intelligence is"
print(f"Prompt: '{prompt}'\n")
_ = real_prefill_decode(qwen_model, tokenizer, prompt, max_new_tokens=30)

Prompt: 'The future of artificial intelligence is'

Prefill: 6 tokens in 466.5 ms
KV-Cache: 0.69 MB (28 layers)
KV shape per layer: K=torch.Size([1, 8, 6, 128]), V=torch.Size([1, 8, 6, 128])
  (batch=1, kv_heads=8, seq_len=6, head_dim=128)

Decode: 30 tokens, avg 24.2 ms/token, total 701.9 ms
Throughput: 42.7 tokens/s

Generated:  a topic that has sparked considerable debate and speculation in the scientific community. As we delve into this subject, it's essential to understand the multifaceted


## 9. Multi-Token Prediction (MTP)

Standard autoregressive generation produces **one token per forward pass**. Multi-Token Prediction (MTP) lets the model predict **multiple future tokens in a single pass**, eliminating the need for a separate draft model (as in speculative decoding).

### Standard vs MTP

| | Standard (Next-Token) | Multi-Token Prediction |
|--|----------------------|------------------------|
| **Training target** | Predict token at $t+1$ only | Predict tokens at $t+1, t+2, \ldots, t+n$ |
| **Prediction heads** | 1 shared `lm_head` | 1 shared trunk + $n$ separate heads |
| **Inference** | 1 token per forward pass | $n$ tokens per forward pass |
| **Draft model needed** | Yes (for speculative decoding) | No (model is its own draft) |
| **Extra memory** | Full draft model | Only $n$ small projection heads |

### Architecture (DeepSeek-V3 style)

```
Input tokens → [Shared Transformer Trunk] → hidden state h_t
                                              ↓
                        ┌─────────────────────┼─────────────────────┐
                        ↓                     ↓                     ↓
                   Head 1 (t+1)          Head 2 (t+2)          Head 3 (t+3)
                   W_proj_1 · h_t        W_proj_2 · h_t        W_proj_3 · h_t
                        ↓                     ↓                     ↓
                   token_t+1             token_t+2             token_t+3
```

Each MTP head is a small projection: `hidden_dim → vocab_size` (same as `lm_head` but with separate weights).
The key insight: predicting $n$ tokens ahead is a **learnable skill** — the model's hidden state encodes enough information to forecast multiple future tokens.

### MTP at Inference Time

1. Run one forward pass through the trunk → get hidden state $h_t$
2. Run all $n$ heads in parallel → get predictions for positions $t+1, t+2, \ldots, t+n$
3. **Accept** tokens greedily from head 1 to head $n$
4. Continue from position $t+n$ (skipping $n-1$ decode steps)

> **Note**: MTP heads need to be trained on multi-token targets. Here we demonstrate the architecture and inference flow with randomly initialized heads.
> In production (DeepSeek-V3), MTP heads are trained end-to-end, achieving ~1.8x speedup.


In [21]:
class MTPModel(nn.Module):
    """Multi-Token Prediction wrapper: adds n prediction heads to a base LLM.

    Each head projects the hidden state to vocabulary logits for a different
    future position (t+1, t+2, ..., t+n).
    """

    def __init__(self, base_model, n_heads=4):
        super().__init__()
        self.base_model = base_model
        self.n_heads = n_heads
        hidden_size = base_model.config.hidden_size
        vocab_size = base_model.config.vocab_size

        # Match base model dtype (e.g., float16)
        model_dtype = next(base_model.parameters()).dtype

        # Shared lm_head for position t+1 (reuse existing)
        # Additional heads for positions t+2, t+3, ..., t+n
        self.mtp_heads = nn.ModuleList([
            nn.Linear(hidden_size, vocab_size, bias=False, dtype=model_dtype)
            for _ in range(n_heads - 1)
        ])

        # Initialize MTP heads with small weights (for demo)
        for head in self.mtp_heads:
            nn.init.normal_(head.weight, std=0.02)

    def forward(self, input_ids, past_key_values=None, use_cache=True):
        """Forward pass: returns logits from all MTP heads."""
        outputs = self.base_model(
            input_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
        )
        hidden_states = outputs.logits  # Actually need hidden states

        # Get hidden states from the model (before lm_head)
        # We need to hook into the model to get hidden states
        return outputs

    def get_mtp_logits(self, hidden_states):
        """Project hidden states through all MTP heads.

        Args:
            hidden_states: (batch, seq_len, hidden_size)
        Returns:
            list of (batch, seq_len, vocab_size) logits, one per head
        """
        # Head 0 uses the base model's lm_head
        head0_logits = self.base_model.lm_head(hidden_states)
        all_logits = [head0_logits]

        # Additional heads
        for head in self.mtp_heads:
            all_logits.append(head(hidden_states))

        return all_logits


# Build MTP model
n_mtp_heads = 4  # Predict tokens at t+1, t+2, t+3, t+4
mtp_model = MTPModel(qwen_model, n_heads=n_mtp_heads).to(device).eval()

mtp_params = sum(p.numel() for p in mtp_model.mtp_heads.parameters())
base_params = sum(p.numel() for p in qwen_model.parameters())
print(f"MTP Model (Qwen3-0.6B + {n_mtp_heads} heads)")
print(f"  Base model params: {base_params:,}")
print(f"  MTP head params:   {mtp_params:,} ({mtp_params/base_params*100:.2f}% of base)")
print(f"  Total params:      {base_params + mtp_params:,}")

MTP Model (Qwen3-0.6B + 4 heads)
  Base model params: 596,049,920
  MTP head params:   466,747,392 (78.31% of base)
  Total params:      1,062,797,312


In [22]:
@torch.no_grad()
def mtp_generate(model, tokenizer, prompt, max_new_tokens=30, n_heads=4):
    """Generate using Multi-Token Prediction.

    Each forward pass produces n_heads predictions for future tokens.
    We accept them greedily (in production, you'd verify/adjust).
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Hook to capture hidden states before lm_head
    hidden_states_cache = {}

    def hook_fn(module, input, output):
        # The last transformer layer output is the hidden states
        hidden_states_cache['hidden'] = input[0] if isinstance(input, tuple) else input

    # Register hook on the norm layer (before lm_head)
    hook = model.base_model.model.norm.register_forward_hook(hook_fn)

    # Prefill
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    outputs = model.base_model(input_ids, use_cache=True)
    past_kv = outputs.past_key_values

    if device.type == "cuda":
        torch.cuda.synchronize()
    prefill_time = time.perf_counter() - t0

    # Get hidden states and run MTP heads
    hidden = hidden_states_cache['hidden']  # (1, seq_len, hidden_size)
    all_logits = model.get_mtp_logits(hidden)  # list of (1, seq_len, vocab_size)

    # Get predictions from each head for the last position
    generated = []
    for head_idx in range(min(n_heads, len(all_logits))):
        token = torch.argmax(all_logits[head_idx][0, -1, :]).item()
        generated.append(token)

    # Decode phase: continue with standard autoregressive for remaining tokens
    decode_times = []
    n_mtp_rounds = 0

    while len(generated) < max_new_tokens:
        # Use the last generated token as input
        new_input = torch.tensor([[generated[-1]]], device=device)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        outputs = model.base_model(new_input, past_key_values=past_kv, use_cache=True)
        past_kv = outputs.past_key_values

        # Get hidden states and run MTP heads
        hidden = hidden_states_cache['hidden']
        all_logits = model.get_mtp_logits(hidden)

        if device.type == "cuda":
            torch.cuda.synchronize()
        decode_times.append(time.perf_counter() - t0)

        # Accept tokens from all heads
        for head_idx in range(min(n_heads, len(all_logits))):
            if len(generated) >= max_new_tokens:
                break
            token = torch.argmax(all_logits[head_idx][0, -1, :]).item()
            generated.append(token)

        n_mtp_rounds += 1

    hook.remove()

    avg_decode = np.mean(decode_times) * 1000 if decode_times else 0
    total_decode = sum(decode_times) * 1000

    print(f"Prefill: {input_ids.shape[1]} tokens in {prefill_time*1000:.1f} ms")
    print(f"Decode: {len(generated)} tokens in {total_decode:.1f} ms ({n_mtp_rounds} MTP rounds)")
    print(f"  Avg per round: {avg_decode:.1f} ms (generates {n_heads} tokens per round)")
    print(f"  Effective throughput: {len(generated)/(total_decode/1000):.1f} tokens/s")

    return generated


# Standard decode for comparison
@torch.no_grad()
def standard_generate(model, prompt, max_new_tokens=30):
    """Standard autoregressive decode (1 token per step)."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    outputs = model.generate(
        input_ids, max_new_tokens=max_new_tokens,
        do_sample=False, use_cache=True
    )

    if device.type == "cuda":
        torch.cuda.synchronize()
    total_time = time.perf_counter() - t0

    generated = outputs[0][input_ids.shape[1]:].tolist()
    print(f"Decode: {len(generated)} tokens in {total_time*1000:.1f} ms")
    print(f"  Throughput: {len(generated)/total_time:.1f} tokens/s")

    return generated

In [23]:
# Compare Standard vs MTP decode

prompt = "The future of artificial intelligence is"
max_gen = 30

print("=== Standard Decode (1 token per step) ===")
std_tokens = standard_generate(qwen_model, prompt, max_gen)
print(f"Output: {tokenizer.decode(std_tokens[:50])}...\n")

print(f"=== MTP Decode ({n_mtp_heads} tokens per step) ===")
mtp_tokens = mtp_generate(mtp_model, tokenizer, prompt, max_gen, n_mtp_heads)
print(f"Output: {tokenizer.decode(mtp_tokens[:50])}...\n")

print("=== Analysis ===")
print(f"Standard: 1 forward pass per token → sequential")
print(f"MTP: 1 forward pass per {n_mtp_heads} tokens → {n_mtp_heads}x fewer passes (theoretical)")
print(f"\nNote: MTP heads here are randomly initialized (not trained).")
print(f"In production (DeepSeek-V3), trained MTP heads achieve ~1.8x real speedup.")
print(f"The speedup comes from reducing decode steps, not from faster computation.")

=== Standard Decode (1 token per step) ===
Decode: 30 tokens in 723.9 ms
  Throughput: 41.4 tokens/s
Output:  a topic that has sparked considerable debate and speculation in the scientific community. As we delve into this subject, it's essential to understand the multifaceted...

=== MTP Decode (4 tokens per step) ===


RuntimeError: expected mat1 and mat2 to have the same dtype, but got: c10::Half != float

### Key Takeaway

At long contexts, **KV-Cache memory** (not model weights or compute) becomes the primary bottleneck for LLM serving. The solutions form a hierarchy:

1. **Architecture-level**: GQA/MQA (already built into Qwen3, LLaMA 3, Mistral)
2. **Quantization**: FP8 KV-Cache (drop-in, production-ready)
3. **Sparse attention**: Only attend to important KV pages (see LLM13)
4. **Memory management**: PagedAttention eliminates fragmentation (see LLM13)
5. **Compression**: MLA (DeepSeek-V3) compresses KV into latent space

These techniques are often combined in production systems (e.g., vLLM uses GQA + PagedAttention + FP8 KV).


## Conclusions

### Technical Concepts Learned

- **Two-Stage Inference**: Prefill (parallel, compute-bound) and Decode (sequential, memory-bound)
- **Arithmetic Intensity**: Prefill is compute-bound (intensity >> 1), Decode is memory-bound (intensity << 1) — decode loads entire model weights for just one token
- **Decoding Strategies**: Greedy, temperature scaling, top-k filtering, top-p (nucleus) sampling
- **Penalty Mechanisms**: Repetition, presence, and frequency penalties to avoid degenerate output
- **FLOPs Analysis**: Over a sequence of length $s$, attention includes an $O(s^2)$ score/mixing term, while MLP scales linearly with $s$ for fixed hidden size. As context grows, attention becomes increasingly expensive.
- **KV-Cache**: Cache K/V tensors from previous tokens to avoid $O(s^2)$ recomputation at each decode step
- **KV-Cache Memory**: GQA (8 KV heads vs 16 Q heads) halves KV-Cache vs MHA; long contexts make KV-Cache the dominant memory consumer
- **Long-Context Solutions**: GQA/MQA, FP8 quantization, sparse attention, PagedAttention, MLA
- **Real Model (Qwen3-0.6B)**: 596M params, GQA 16/8, head_dim=128, max 40K context
- **Multi-Token Prediction (MTP)**: Predict multiple future tokens per forward pass — no separate draft model needed, only adds small projection heads (~5% extra params)

### Experiment Further

- Implement beam search with length penalty
- Train MTP heads on real data and measure actual speedup vs. random heads
- Implement FP8 quantized KV-Cache and measure quality degradation
- Measure how KV-Cache memory scales with batch size in a serving scenario
- Compare streaming generation latency (time-to-first-token vs. throughput)
- Explore MLA (Multi-head Latent Attention) from DeepSeek-V3

---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
